In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [13]:
df = pd.read_csv("weatherAUS.csv")
df.head()

,Date,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,2008-12-01,Albury,13.4,22.9,0.6,NaN,NaN,W,44.0,W,...,71.0,22.0,1007.7,1007.1,8.0,NaN,16.9,21.8,No,No
1,2008-12-02,Albury,7.4,25.1,0.0,NaN,NaN,WNW,44.0,NNW,...,44.0,25.0,1010.6,1007.8,NaN,NaN,17.2,24.3,No,No
2,2008-12-03,Albury,12.9,25.7,0.0,NaN,NaN,WSW,46.0,W,...,38.0,30.0,1007.6,1008.7,NaN,2.0,21.0,23.2,No,No
3,2008-12-04,Albury,9.2,28.0,0.0,NaN,NaN,NE,24.0,SE,...,45.0,16.0,1017.6,1012.8,NaN,NaN,18.1,26.5,No,No
4,2008-12-05,Albury,17.5,32.3,1.0,NaN,NaN,W,41.0,ENE,...,82.0,33.0,1010.8,1006.0,7.0,8.0,17.8,29.7,No,No


In [14]:
df.shape

(145460, 23)

## Step 1 — Drop rows where the target `RainTomorrow` is null

These rows have no label, so they can't be used for training or evaluation. We drop them up front (this also removes most rows where `RainToday`/`Rainfall` are missing, since they overlap heavily).

In [15]:
# Step 1 — Drop rows where the target RainTomorrow is null
before = len(df)
df = df.dropna(subset=['RainTomorrow']).reset_index(drop=True)
print(f"Dropped {before - len(df):,} rows with null RainTomorrow; {len(df):,} rows remain")
print(df['RainTomorrow'].value_counts(dropna=False))

Dropped 3,267 rows with null RainTomorrow; 142,193 rows remain
RainTomorrow
No     110316
Yes     31877
Name: count, dtype: int64


## Step 2 — Chronological train / validation / test split

We split **before** any imputation or scaling so that statistics learned later (medians, means, scaler) come only from the training set — no leakage from the future.

- **Train:** 2008–2014
- **Validation:** 2015
- **Test:** 2016 and later

In [16]:
# Step 2 — Chronological train / validation / test split
# Parse Date and split by year so no future information leaks into the past.
df['Date'] = pd.to_datetime(df['Date'])
df['Year']  = df['Date'].dt.year
df['Month'] = df['Date'].dt.month

df_train = df[df['Year'] <= 2014].copy()   # 2008-2014  -> training
df_val   = df[df['Year'] == 2015].copy()   # 2015       -> validation
df_test  = df[df['Year'] >= 2016].copy()   # 2016+      -> test

for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    yrs = f"{part['Year'].min()}-{part['Year'].max()}"
    pos = (part['RainTomorrow'] == 'Yes').mean()
    print(f"{name:5s}: {len(part):>7,} rows  years {yrs}  RainTomorrow=Yes {pos:.1%}")

train:  98,988 rows  years 2007-2014  RainTomorrow=Yes 22.5%
val  :  17,231 rows  years 2015-2015  RainTomorrow=Yes 21.2%
test :  25,974 rows  years 2016-2017  RainTomorrow=Yes 22.9%


## Step 3 — Wind direction cyclical (sin/cos) encoding

Wind directions are a 16-point compass (N, NNE, ... NNW). Treating them as unordered categories loses the fact that N and NNW are adjacent. We map each direction to its compass bearing in degrees, then to `sin` and `cos`, giving 2 numeric features per column that wrap around correctly. Done for all three: `WindGustDir`, `WindDir9am`, `WindDir3pm`.

In [ ]:
# Step 3 — Wind direction cyclical (sin/cos) encoding for all 3 direction columns
# 16-point compass -> degrees -> radians -> (sin, cos).
# sin/cos keeps adjacent directions close (e.g. N and NNW) and is a deterministic
# transform (no fitting), so there is no train/test leakage in applying it per split.

COMPASS = {
    'N': 0.0,   'NNE': 22.5,  'NE': 45.0,   'ENE': 67.5,
    'E': 90.0,  'ESE': 112.5, 'SE': 135.0,  'SSE': 157.5,
    'S': 180.0, 'SSW': 202.5, 'SW': 225.0,  'WSW': 247.5,
    'W': 270.0, 'WNW': 292.5, 'NW': 315.0,  'NNW': 337.5,
}

WIND_DIR_COLS = ['WindGustDir', 'WindDir9am', 'WindDir3pm']

def encode_wind_dirs(frame, cols=WIND_DIR_COLS):
    """Return a copy of `frame` with *_sin / *_cos columns added for each wind-direction column.
    NaN directions stay NaN in the sin/cos columns and are handled later during imputation."""
    out = frame.copy()
    for c in cols:
        rad = np.deg2rad(out[c].map(COMPASS).astype('float64'))
        out[c + '_sin'] = np.sin(rad)
        out[c + '_cos'] = np.cos(rad)
    return out

df_train = encode_wind_dirs(df_train)
df_val   = encode_wind_dirs(df_val)
df_test  = encode_wind_dirs(df_test)

# Sanity check: show the original + encoded columns for one wind field
df_train[['WindGustDir', 'WindGustDir_sin', 'WindGustDir_cos']].head()

## Step 4 — Encode binary Yes/No columns

`RainToday` (a feature) and `RainTomorrow` (the target) are `Yes`/`No` strings. We map them to `1`/`0`. The target has no nulls left (dropped in Step 1); `RainToday` still has a few nulls, which stay `NaN` for now and get handled in the imputation step.

In [17]:
# Step 4 — Encode binary Yes/No columns to 1/0
yn_map = {'Yes': 1, 'No': 0}

for part in (df_train, df_val, df_test):
    part['RainToday']    = part['RainToday'].map(yn_map)
    part['RainTomorrow'] = part['RainTomorrow'].map(yn_map)

# Check: target is fully 0/1 (no NaN), RainToday may still have a few NaN to impute later
print("RainTomorrow (train) value counts:")
print(df_train['RainTomorrow'].value_counts(dropna=False))
print("\nRainToday (train) null count:", df_train['RainToday'].isna().sum())

RainTomorrow (train) value counts:
RainTomorrow
0    76705
1    22283
Name: count, dtype: int64

RainToday (train) null count: 1000


## Step 5 — Impute temperature and rainfall columns

Fill missing temperature and rainfall values using a group statistic learned from the **training set only**, with a fallback chain so every null gets filled:

**`(Location, Month)` → `Location` → global**

- **Temperature** (`MinTemp`, `MaxTemp`, `Temp9am`, `Temp3pm`) → **mean** (roughly symmetric).
- **Rainfall** → **median** (right-skewed; mean is distorted by storm days).

No missing-indicator flags here — these columns are ≤2.5% missing, so a flag would be almost all zeros. Flags are reserved for the high-missing columns (Sunshine / Evaporation / Cloud) in a later step.

In [18]:
# Step 5 — Impute temperature and rainfall columns
# Group statistic learned from TRAIN ONLY, applied to all splits (no leakage).
# Fallback chain: (Location, Month) -> Location -> global.

def fit_group_stats(train, col, agg):
    """Learn the imputation statistic at three granularities from the training set."""
    lm   = train.groupby(['Location', 'Month'])[col].agg(agg)  # per (Location, Month)
    loc  = train.groupby('Location')[col].agg(agg)             # per Location
    glob = train[col].agg(agg)                                 # global scalar
    return lm, loc, glob

def impute_col(frame, col, stats):
    """Fill NaNs in frame[col] using the fallback chain (Location, Month) -> Location -> global."""
    lm, loc, glob = stats
    idx = pd.MultiIndex.from_frame(frame[['Location', 'Month']])
    fill_lm  = pd.Series(lm.reindex(idx).to_numpy(), index=frame.index)
    fill_loc = frame['Location'].map(loc)
    return frame[col].fillna(fill_lm).fillna(fill_loc).fillna(glob)

# column -> aggregation: temperature uses mean, rainfall uses median
impute_plan = {
    'MinTemp': 'mean', 'MaxTemp': 'mean', 'Temp9am': 'mean', 'Temp3pm': 'mean',
    'Rainfall': 'median',
}

for col, agg in impute_plan.items():
    stats = fit_group_stats(df_train, col, agg)   # fit on train
    for part in (df_train, df_val, df_test):      # apply to all splits
        part[col] = impute_col(part, col, stats)

# Verify: no nulls remain in the imputed columns across any split
cols = list(impute_plan)
print("Remaining nulls after imputation:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s}", dict(part[cols].isna().sum()))

Remaining nulls after imputation:
  train {'MinTemp': np.int64(0), 'MaxTemp': np.int64(0), 'Temp9am': np.int64(0), 'Temp3pm': np.int64(0), 'Rainfall': np.int64(0)}
  val   {'MinTemp': np.int64(0), 'MaxTemp': np.int64(0), 'Temp9am': np.int64(0), 'Temp3pm': np.int64(0), 'Rainfall': np.int64(0)}
  test  {'MinTemp': np.int64(0), 'MaxTemp': np.int64(0), 'Temp9am': np.int64(0), 'Temp3pm': np.int64(0), 'Rainfall': np.int64(0)}


## Step 6 — Impute the high-missing columns (median) + missing-indicator flags

`Sunshine` (48%), `Evaporation` (43%), `Cloud3pm` (41%), `Cloud9am` (38%) are missing for large stretches (often whole stations/periods), so we treat them specially:

1. **Flag first** — add `<col>_was_missing` (1 where the original value was NaN, else 0), computed **before** imputing. This is a per-row check, so doing it on val/test is not leakage. The flag becomes a feature, letting the model use the *fact* a reading was missing.
2. **Then impute** the value with the **median** (these are skewed / bounded-okta columns), using the same train-fit fallback chain `(Location, Month)` → `Location` → global from Step 5.

Reuses `fit_group_stats` and `impute_col` defined in Step 5.

In [19]:
# Step 6 — Impute high-missing columns (median) + missing-indicator flags
high_missing = ['Sunshine', 'Evaporation', 'Cloud9am', 'Cloud3pm']

# 1) Flag BEFORE imputing — captures the original missingness pattern (per-row, no leakage)
for part in (df_train, df_val, df_test):
    for col in high_missing:
        part[col + '_was_missing'] = part[col].isna().astype(int)

# 2) Impute the value with the median; fit on TRAIN, apply to all splits
#    (reuses fit_group_stats / impute_col from Step 5; fallback (Loc,Month) -> Loc -> global)
for col in high_missing:
    stats = fit_group_stats(df_train, col, 'median')
    for part in (df_train, df_val, df_test):
        part[col] = impute_col(part, col, stats)

# Verify: no nulls remain, and show how much of each column was flagged as missing
flag_cols = [c + '_was_missing' for c in high_missing]
print("Remaining nulls after imputation:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s}", dict(part[high_missing].isna().sum()))

print("\nShare flagged as missing (train):")
print(df_train[flag_cols].mean().round(3).to_string())

Remaining nulls after imputation:
  train {'Sunshine': np.int64(0), 'Evaporation': np.int64(0), 'Cloud9am': np.int64(0), 'Cloud3pm': np.int64(0)}
  val   {'Sunshine': np.int64(0), 'Evaporation': np.int64(0), 'Cloud9am': np.int64(0), 'Cloud3pm': np.int64(0)}
  test  {'Sunshine': np.int64(0), 'Evaporation': np.int64(0), 'Cloud9am': np.int64(0), 'Cloud3pm': np.int64(0)}

Share flagged as missing (train):
Sunshine_was_missing       0.411
Evaporation_was_missing    0.375
Cloud9am_was_missing       0.361
Cloud3pm_was_missing       0.371


## Step 7 — Fill remaining `RainToday` nulls

`RainToday` describes the same event as the previous day's `RainTomorrow`, so we fill it with a cascade:

1. **Borrow `RainTomorrow[D−1]`** for the same Location when day D−1 is present and consecutive (a real label, verified to match 100% of the time). In this dataset the missing `RainToday` rows sit at date gaps, so this fills ~0 — but it's the most authentic source where it applies.
2. **Fallback — derive from `Rainfall`:** the dataset defines `RainToday = 1` iff `Rainfall > 1 mm` (verified to reproduce the label 100%). Since `Rainfall` was imputed in Step 5, this fills every remaining null.

In [20]:
# Step 7 — Fill remaining RainToday nulls
for part in (df_train, df_val, df_test):
    part.sort_values(['Location', 'Date'], inplace=True)
    prev_rt   = part.groupby('Location')['RainTomorrow'].shift(1)
    prev_date = part.groupby('Location')['Date'].shift(1)
    consecutive = (part['Date'] - prev_date).dt.days == 1
    # (a) authentic: previous-day RainTomorrow on consecutive days
    part['RainToday'] = part['RainToday'].fillna(prev_rt.where(consecutive))
    # (b) fallback: dataset definition RainToday = 1 iff Rainfall > 1mm (Rainfall imputed in Step 5)
    part['RainToday'] = part['RainToday'].fillna((part['Rainfall'] > 1).astype(int)).astype(int)

print("RainToday nulls after fill:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s}", int(part['RainToday'].isna().sum()))

RainToday nulls after fill:
  train 0
  val   0
  test  0


## Step 8 — Impute remaining numeric columns (Pressure, Humidity, WindSpeed, WindGustSpeed)

`Pressure9am/3pm`, `Humidity9am/3pm`, `WindGustSpeed`, `WindSpeed9am/3pm` are all low-missingness (≤10%), so no flags. Impute with the **median** via the same train-fit fallback chain `(Location, Month)` → `Location` → global (median is safe for the mildly-skewed wind/pressure/humidity distributions). Reuses `fit_group_stats` / `impute_col` from Step 5.

In [21]:
# Step 8 — Impute remaining numeric columns (median, no flags)
median_cols = ['Pressure9am', 'Pressure3pm', 'Humidity9am', 'Humidity3pm',
               'WindGustSpeed', 'WindSpeed9am', 'WindSpeed3pm']

for col in median_cols:
    stats = fit_group_stats(df_train, col, 'median')   # fit on train
    for part in (df_train, df_val, df_test):           # apply to all splits
        part[col] = impute_col(part, col, stats)

print("Remaining nulls after Step 8:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s}", dict(part[median_cols].isna().sum()))

Remaining nulls after Step 8:
  train {'Pressure9am': np.int64(0), 'Pressure3pm': np.int64(0), 'Humidity9am': np.int64(0), 'Humidity3pm': np.int64(0), 'WindGustSpeed': np.int64(0), 'WindSpeed9am': np.int64(0), 'WindSpeed3pm': np.int64(0)}
  val   {'Pressure9am': np.int64(0), 'Pressure3pm': np.int64(0), 'Humidity9am': np.int64(0), 'Humidity3pm': np.int64(0), 'WindGustSpeed': np.int64(0), 'WindSpeed9am': np.int64(0), 'WindSpeed3pm': np.int64(0)}
  test  {'Pressure9am': np.int64(0), 'Pressure3pm': np.int64(0), 'Humidity9am': np.int64(0), 'Humidity3pm': np.int64(0), 'WindGustSpeed': np.int64(0), 'WindSpeed9am': np.int64(0), 'WindSpeed3pm': np.int64(0)}


## Step 9 — Fill wind-direction sin/cos nulls, drop raw direction strings

The raw `WindGustDir` / `WindDir9am` / `WindDir3pm` string columns are now redundant — their information lives in the `_sin`/`_cos` pairs from Step 3. We:

1. **Fill the `_sin`/`_cos` nulls with 0.** For a direction, `(sin, cos) = (0, 0)` is the zero-length wind vector — the natural neutral for "no/unknown prevailing direction", and it treats every missing direction identically instead of nudging it toward some median bearing.
2. **Drop the raw string columns** so only the numeric encodings remain.

After this the dataset is fully null-free.

In [22]:
# Step 9 — Fill wind-direction sin/cos nulls with 0, drop raw string columns
sincos_cols = [c + s for c in WIND_DIR_COLS for s in ('_sin', '_cos')]

for part in (df_train, df_val, df_test):
    part[sincos_cols] = part[sincos_cols].fillna(0)
    part.drop(columns=WIND_DIR_COLS, inplace=True)

# Final check: confirm the whole dataset is null-free across all splits
print("Total remaining nulls per split:")
for name, part in [('train', df_train), ('val', df_val), ('test', df_test)]:
    print(f"  {name:5s} {int(part.isna().sum().sum())} nulls,  shape {part.shape}")

NameError: name 'WIND_DIR_COLS' is not defined